In [1]:
import pandas as pd

In [2]:
stops_df = pd.read_csv('BCT-data/BCTstops.txt')
stops_df = stops_df.drop(columns=['wheelchair_boarding', 'stop_code'])
stops_df.to_csv('converted-data/stops.csv', index=False)

In [ ]:
routes_df = pd.read_csv('BCT-data/BCTroutes.txt')
routes_df['route_name'] = routes_df['route_short_name'] + ': ' + routes_df['route_long_name']
routes_df = stops_df.drop(columns=['route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color'])
routes_df.to_csv('converted-data/routes.csv', index=False)

KeyError: "['route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color'] not found in axis"

In [10]:
trips_df = pd.read_csv('BCT-data/BCTtrips.txt')
trips_df = routes_df.drop_duplicates(subset='route_id')
# naive way of assigning one representative trip to each route
print(trips_df.info())
trips_df.head()


<class 'pandas.DataFrame'>
Index: 69 entries, 0 to 10709
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   route_id       69 non-null     str  
 1   service_id     69 non-null     int64
 2   trip_id        69 non-null     str  
 3   trip_headsign  69 non-null     str  
 4   shape_id       69 non-null     int64
 5   block_id       69 non-null     int64
 6   direction_id   69 non-null     int64
dtypes: int64(4), str(3)
memory usage: 4.3 KB
None


,route_id,service_id,trip_id,trip_headsign,shape_id,block_id,direction_id
0,40-VIC,5462,11918189:14530255:14594817,UVic via Admirals/McKenzie,50587,14594817,0
1,4-VIC,5473,12337295:14530260:14827853,UVic via Hillside,50582,14827853,0
2,5-VIC,5473,12337224:14530261:14829118,James Bay - Fisherman's Wharf,50815,14829118,1
3,7N-VIC,5467,11908246:14530262:14591278,Drop Off Only,50567,14591278,0
5,95-VIC,5475,12358351:14530265:14827197,Blink - to Langford,54967,14827197,1


In [19]:
route_stops = pd.read_csv('BCT-data/BCTstop_times.txt')
route_stops_filtered = route_stops[route_stops['trip_id'].isin(trips_df['trip_id'])]
route_stops_filtered

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,shape_dist_traveled,stop_headsign,pickup_type,drop_off_type,timepoint
0,11918189:14530255:14594817,11:40:00,11:40:00,100698,1,0,NaN,0,0,1
1,11918189:14530255:14594817,11:40:00,11:40:00,100693,2,252,NaN,0,0,0
2,11918189:14530255:14594817,11:41:00,11:41:00,100635,3,640,NaN,0,0,0
3,11918189:14530255:14594817,11:42:00,11:42:00,100594,4,850,NaN,0,0,0
4,11918189:14530255:14594817,11:43:00,11:43:00,100570,5,1550,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...
407076,12358849:14548460:14828453,11:00:00,11:00:00,102344,28,18468,NaN,0,0,0
407077,12358849:14548460:14828453,11:01:00,11:01:00,102332,29,18794,NaN,0,0,0
407078,12358849:14548460:14828453,11:01:00,11:01:00,102308,30,19222,NaN,0,0,0
407079,12358849:14548460:14828453,11:02:00,11:02:00,102299,31,19820,NaN,0,0,0


In [20]:
route_id_map = trips_df.set_index('trip_id')['route_id']
route_stops_filtered['route_id'] = route_stops_filtered['trip_id'].map(route_id_map)

In [28]:
route_stops_filtered[['stop_id', 'route_id', 'stop_sequence']].to_csv('converted-data/route_stops.csv', index=False)